# E-to-E backbone method comparison

This notebook treats `mij_EE_matrix.csv` as a directed weighted graph over the excitatory nodes only. The matrix was derived from `mij_matrix.csv` by restricting rows and columns to excitatory senders, then this notebook keeps positive off-diagonal excitatory-to-excitatory transition weights as candidate path edges. That filtering step is intentional: the goal is not to describe every signed connection in the original matrix, but to compare candidate excitatory backbone paths under one shared positive-weight objective.

The notebook compares six backbone-building methods from every excitatory seed:

1. greedy tree/path using the largest outgoing weights to other excitatory neurons  
2. maximum spanning tree-derived path  
3. branch-and-bound search  
4. dynamic programming / beam-limited DP  
5. mixed-integer linear programming with subtour elimination  
6. maximum-weight asymmetric Hamiltonian path  

These methods are deliberately different in search style. Greedy construction is a fast local baseline, the maximum spanning tree gives a global non-cycling skeleton, branch-and-bound provides a bounded exhaustive search, dynamic programming gives either an exact subset-state solution or a tractable beam-limited approximation, MILP directly optimizes the unconstrained simple-path objective with linear constraints that prevent subtours, and the Hamiltonian variant asks what happens when every excitatory node must appear exactly once. Keeping all methods in the same notebook makes disagreements easier to inspect, because every result is scored with the same transition matrix, seed definitions, diagonal handling, and output schema.

Reusable implementation lives in `ee_backbone_analysis.py` so this notebook stays focused on configuration, execution, and review. Diagonal entries are ignored here; if a future dataset needs special self-connection handling, handle that as part of dataset preparation before running the analysis.


In [ ]:
from pathlib import Path

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

import pandas as pd

from ee_backbone_analysis import (
    EPS,
    beam_dp_path,
    best_mst_path_from_seed,
    branch_bound_path,
    build_disagreement_table,
    build_milp_benchmark_table,
    build_tree_adjacency,
    edges_from_path,
    exact_bitmask_dp_path,
    greedy_path,
    maximum_spanning_tree_edges,
    maximum_weight_asymmetric_hamiltonian_path,
    milp_subtour_elimination_path,
    path_to_records,
    prepare_excitatory_matrix_data,
    write_comparison_outputs,
    write_setup_outputs,
)

INPUT_CSV = Path("matrices/mij_EE_matrix.csv")
OUTDIR = Path("ee_backbone_comparison_outputs")

# Branch-and-bound guardrails. Increase for deeper search.
BB_TIME_LIMIT_PER_SEED = 0.25
BB_NODE_LIMIT_PER_SEED = 50_000

# DP guardrails.
NOTEBOOK_EXACT_DP_MAX_E_NODES = 22
BEAM_WIDTH = 5_000

# MILP guardrails. Increase if any seed returns a non-optimal solver status.
MILP_TIME_LIMIT_PER_SEED = 30
HAMILTONIAN_TIME_LIMIT_PER_SEED = 30


## Prepare data

This section builds the graph that all methods use from `mij_EE_matrix.csv`. Because that file already contains only excitatory rows and columns, the preparation step treats every matrix label as an excitatory node instead of re-inferring excitatory status from the restricted matrix. It then clips the working transition matrix to positive weights and removes the diagonal.

Negative and zero transitions are excluded from the candidate graph because the backbone methods below all optimize a positive path score. The diagonal is set to zero before the methods run, so every selected edge represents a transition between two distinct excitatory nodes. The setup outputs are written before any method-specific search begins; those files make the retained node list and edge set auditable if a later result looks surprising.


In [ ]:
data = prepare_excitatory_matrix_data(
    INPUT_CSV,
    eps=EPS,
)
write_setup_outputs(data, OUTDIR)

print(f"E-to-E matrix: {data.full_matrix.shape[0]} x {data.full_matrix.shape[1]}")
print(f"Excitatory nodes retained: {len(data.excitatory_nodes)}")
print(f"Positive E-to-E transitions retained: {(data.transition_weights > EPS).sum()}")
print(f"Positive directed E-to-E transition edges retained: {sum(len(v) for v in data.adjacency.values())}")


## Method sections

Each method below has its own reasoning note and code cell. The methods are arranged from simpler heuristics to more global searches so the reader can see how much extra complexity changes the selected backbones.

The important comparison point is that all methods receive the same prepared E-to-E graph and report through the same record-building helpers. That keeps the interpretation clean: differences in path length, terminal node, or summed weight should come from the search strategy itself, not from different preprocessing or scoring rules. The separate result tables are combined afterward for output files, disagreement checks, and plots.


## Greedy Tree/Path

The greedy tree/path method starts at each excitatory seed and repeatedly chooses the largest positive outgoing transition to an unvisited excitatory neuron. This is the simplest local rule: at every step it asks, "which reachable excitatory neuron has the strongest direct weight from the current node?" The path stops when there are no positive outgoing transitions to unvisited excitatory nodes. Diagonal/self entries are ignored; scoring is based on transitions between distinct excitatory neurons.

This method is useful because it gives a transparent baseline for the rest of the notebook. If a strong backbone is dominated by a sequence of locally obvious transitions, the greedy path will often recover it quickly and with very little tuning. It also makes failure modes easy to diagnose: when greedy performs poorly, the reason is usually that an early high-weight edge leads into a dead end or prevents access to a better downstream sequence. That makes it a good reference point for judging whether the more expensive searches are adding real value.


In [ ]:
greedy_results = []
greedy_edges = []

for seed in range(len(data.excitatory_nodes)):
    path = greedy_path(seed, data)
    method = "greedy_tree_path"
    greedy_results.append(path_to_records(seed, method, path, data))
    greedy_edges.extend(edges_from_path(seed, method, path, data))

greedy_df = pd.DataFrame(greedy_results)
greedy_df.head()


## Maximum Spanning Tree

The maximum spanning tree method converts the directed transition matrix into an undirected topology by taking the stronger direction for each pair of excitatory neurons. It then builds the tree that keeps the largest total set of non-cycling connections. For each seed, the reported backbone is the highest-scoring path through that tree. This gives a global skeleton of strong excitatory connectivity, while reported path scores use the original directed transition weights between distinct nodes.

This works as a backbone heuristic because a spanning tree removes cycles and redundant alternatives while preserving a connected set of high-weight relationships. It is not trying to solve the same directed simple-path problem exactly; instead, it asks which strong pairwise links survive when the graph is forced into a sparse, interpretable skeleton. The seed-specific path is then chosen inside that skeleton, which can reveal broad corridors of connectivity even when the full directed graph has many competing edges. The tradeoff is that directionality influences scoring but not tree construction, so the method is best read as a structural comparison rather than a directed optimum.


In [ ]:
mst_edges = maximum_spanning_tree_edges(data.transition_weights)
tree_adj = build_tree_adjacency(len(data.excitatory_nodes), mst_edges)

mst_results = []
mst_path_edges = []
for seed in range(len(data.excitatory_nodes)):
    path = best_mst_path_from_seed(seed, tree_adj, data)
    method = "maximum_spanning_tree"
    mst_results.append(path_to_records(seed, method, path, data))
    mst_path_edges.extend(edges_from_path(seed, method, path, data))

mst_df = pd.DataFrame(mst_results)
mst_df.head()


## Branch-and-Bound

Branch-and-bound searches the space of simple directed paths from each seed while keeping the best path found so far. At each partial path, it estimates an optimistic upper bound on how much more score could still be gained; if that bound cannot beat the current best path, the branch is pruned. This gives a more exhaustive search than greedy methods, with time and node-count guardrails to keep the run practical.

The key idea is that many partial paths can be ruled out without explicitly completing them. If even the best possible extension of a partial path cannot exceed the incumbent score, every longer path under that branch is irrelevant for the current objective. That pruning makes an otherwise explosive simple-path search more manageable while preserving exactness when the search finishes completely. The notebook records both the number of states visited and the stop status, because a result stopped by `time_limit` or `node_limit` should be read as the best path found under the guardrails rather than a proven optimum.


In [ ]:
bb_results = []
bb_edges = []

for seed in range(len(data.excitatory_nodes)):
    path, score, states, status = branch_bound_path(
        seed,
        data,
        time_limit=BB_TIME_LIMIT_PER_SEED,
        node_limit=BB_NODE_LIMIT_PER_SEED,
    )
    method = "branch_and_bound"
    bb_results.append(
        path_to_records(
            seed,
            method,
            path,
            data,
            {
                "bb_states_seen": states,
                "bb_status": status,
                "bb_score": score,
            },
        )
    )
    bb_edges.extend(edges_from_path(seed, method, path, data))

bb_df = pd.DataFrame(bb_results)
bb_df.head()


## Dynamic Programming

Dynamic programming tracks the best score for path states defined by visited-node sets and the current terminal node. When the excitatory set is small enough, this gives an exact bitmask dynamic program over simple directed paths: each state says which nodes have already been used and where the path currently ends, and the recurrence extends that state by one valid unvisited outgoing neighbor. Because the best score for a state summarizes all earlier ways of reaching the same visited set and terminal node, dominated histories do not need to be kept.

For larger excitatory sets, the number of possible visited sets grows exponentially, so the notebook switches to a beam-limited dynamic program that keeps only the strongest states at each expansion depth. The beam version preserves the same state logic but intentionally trades exactness for tractability. That means it can return lower path weights than other methods when the beam prunes states that look weaker early but would have led to stronger later continuations. In this notebook, the DP result should therefore be read as exact only when `dp_mode == "exact"`; when `dp_mode == "beam"`, it is a heuristic comparison point rather than the final maximizing benchmark.

A closely related computation appears in Held and Karp's dynamic programming approach to sequencing problems, which also indexes subproblem values by a subset of visited items and a terminal item. The objective here is adapted to a best positive directed path rather than a tour, but the state representation is the same core idea.


In [ ]:
dp_results = []
dp_edges = []

for seed in range(len(data.excitatory_nodes)):
    if len(data.excitatory_nodes) <= NOTEBOOK_EXACT_DP_MAX_E_NODES:
        path, score, mode = exact_bitmask_dp_path(seed, data)
    else:
        path, score, mode = beam_dp_path(seed, data, beam_width=BEAM_WIDTH)
    method = "dynamic_programming"
    dp_results.append(
        path_to_records(
            seed,
            method,
            path,
            data,
            {
                "dp_mode": mode,
                "dp_score": score,
            },
        )
    )
    dp_edges.extend(edges_from_path(seed, method, path, data))

dp_df = pd.DataFrame(dp_results)
dp_df.head()


## MILP with Subtour Elimination

The mixed-integer linear program directly models the maximum-weight simple directed path from each seed. It uses a binary variable for each allowed directed edge, a binary variable for each included node, and an order variable for each node. The objective maximizes the sum of selected positive transition weights.

The degree constraints force the selected edges to form one path that starts at the seed: the seed has no incoming edge, every other selected node has exactly one incoming edge, and every selected node has at most one outgoing edge. The MTZ-style order constraints provide subtour elimination. If edge `i -> j` is selected, then `j` must appear later than `i` in the path order; a directed cycle would require the order numbers to increase all the way around the cycle and return to the starting node, which is impossible.

This is the most direct optimizer in the notebook. When the solver status is `Optimal`, the reported path is the exact maximum-weight simple path under this positive E-to-E edge set and diagonal handling. If any seed returns a non-optimal status, increase `MILP_TIME_LIMIT_PER_SEED` and rerun before treating that seed as solved.


In [ ]:
milp_results = []
milp_edges = []

for seed in range(len(data.excitatory_nodes)):
    path, score, status, objective = milp_subtour_elimination_path(
        seed,
        data,
        time_limit=MILP_TIME_LIMIT_PER_SEED,
    )
    method = "milp_subtour_elimination"
    milp_results.append(
        path_to_records(
            seed,
            method,
            path,
            data,
            {
                "milp_status": status,
                "milp_score": score,
                "milp_objective": objective,
                "milp_time_limit_seconds": MILP_TIME_LIMIT_PER_SEED,
            },
        )
    )
    milp_edges.extend(edges_from_path(seed, method, path, data))

milp_df = pd.DataFrame(milp_results)
milp_df.head()


## Maximum-Weight Asymmetric Hamiltonian Path

The maximum-weight asymmetric Hamiltonian path method is another MILP, but it adds a stricter requirement than the unconstrained simple-path MILP above: every excitatory node must be visited exactly once. The graph remains asymmetric because the selected edge `i -> j` and the reverse edge `j -> i` can have different weights, and the solver is free to choose whichever directed ordering maximizes total transition weight from the fixed seed.

The formulation uses one binary variable per allowed directed edge and one order variable per node. Each non-seed node must have exactly one incoming edge, the seed has no incoming edge, every node has at most one outgoing edge, and the total number of selected edges is `n - 1`. MTZ-style order constraints eliminate subtours by forcing each selected edge to move forward in the path order.

This method answers a different question from the previous MILP. The unconstrained MILP can stop before visiting all nodes if adding another node would require a weak continuation, while the Hamiltonian path is forced to include all 32 excitatory nodes. Because of that forced coverage, it can be biologically useful as a complete ordering, but it should not be expected to dominate the unconstrained MILP on summed path weight.


In [ ]:
hamiltonian_results = []
hamiltonian_edges = []

for seed in range(len(data.excitatory_nodes)):
    path, score, status, objective = maximum_weight_asymmetric_hamiltonian_path(
        seed,
        data,
        time_limit=HAMILTONIAN_TIME_LIMIT_PER_SEED,
    )
    method = "maximum_weight_asymmetric_hamiltonian_path"
    hamiltonian_results.append(
        path_to_records(
            seed,
            method,
            path,
            data,
            {
                "hamiltonian_status": status,
                "hamiltonian_score": score,
                "hamiltonian_objective": objective,
                "hamiltonian_time_limit_seconds": HAMILTONIAN_TIME_LIMIT_PER_SEED,
            },
        )
    )
    hamiltonian_edges.extend(edges_from_path(seed, method, path, data))

hamiltonian_df = pd.DataFrame(hamiltonian_results)
hamiltonian_df.head()


## Combine method results

This section stacks the per-method result tables into one long comparison table and combines all selected edges into a single edge table. The long table is the most convenient shape for filtering by seed or method, while the edge table preserves the actual step-by-step transitions selected by each search.

Combining results only after every method has run avoids accidental cross-method dependencies. It also lets downstream outputs use one shared schema for path length, node count, terminal node, summed weight, and method-specific metadata such as branch-and-bound stop status, DP mode, MILP solver status, or Hamiltonian solver status.


In [ ]:
comparison = pd.concat(
    [greedy_df, mst_df, bb_df, dp_df, milp_df, hamiltonian_df],
    ignore_index=True,
)
edge_table = pd.DataFrame(
    greedy_edges + mst_path_edges + bb_edges + dp_edges + milp_edges + hamiltonian_edges
)

comparison.sort_values(["seed", "method"]).reset_index(drop=True).head(20)


## Write outputs

This section writes the comparison artifacts to `ee_backbone_comparison_outputs`. The long file keeps one row per seed-method result, the wide file places methods side by side for each seed, the edge file records selected transitions, the method summary aggregates performance across seeds, the branch-and-bound/DP disagreement file preserves the earlier diagnostic comparison, and the MILP benchmark file measures each non-benchmark method's gap to the unconstrained MILP solution.

Writing these files makes the notebook useful as both an interactive analysis and a reproducible handoff artifact. A reader can inspect the notebook narrative, rerun the computations, or load the CSV outputs directly into another plotting or review workflow without reimplementing the method comparison.


In [ ]:
wide, method_summary = write_comparison_outputs(comparison, edge_table, OUTDIR)

print("Wrote:")
for name in [
    "method_comparison_long.csv",
    "method_comparison_wide.csv",
    "all_methods_edges.csv",
    "method_summary.csv",
    "bb_dp_disagreements.csv",
    "milp_benchmark_gaps.csv",
]:
    print(f"  {OUTDIR / name}")

method_summary


## MILP benchmark gaps

This section compares every non-benchmark method against the unconstrained MILP result for the same seed. When `milp_status == "Optimal"`, the `weight_gap_to_milp` column is the amount of path weight left on the table under the same positive E-to-E graph.

This view is more useful than treating the dynamic-programming output as the benchmark when the notebook is in beam mode. Beam DP is a heuristic at this graph size, while the unconstrained MILP provides an exact reference whenever the solver proves optimality. The Hamiltonian method is included in the gap table too, but its gap should be interpreted as the cost of forcing complete node coverage rather than as a failure to solve the same unconstrained problem.


In [ ]:
milp_benchmark = build_milp_benchmark_table(comparison)
milp_benchmark.sort_values("weight_gap_to_milp", ascending=False).head(20)


## Path weights by method

This plot compares the distribution of `summed_weight` across methods. The purpose is to show the method-level score profile at a glance: whether one method consistently finds heavier paths, whether a heuristic has high variance across seeds, or whether several approaches land in a similar score range.

The boxplot is a compact summary rather than a replacement for seed-level review. With both MILP methods included, the plot should be read alongside the benchmark gaps and path lengths: the unconstrained MILP is the maximizing benchmark, while the Hamiltonian method is a complete 32-node ordering with a stricter coverage requirement.


In [ ]:
if plt is None:
    print("matplotlib is not installed; skipping boxplot.")
else:
    plot_df = comparison[["method", "summed_weight"]].copy()
    ax = plot_df.boxplot(column="summed_weight", by="method", rot=45)
    plt.suptitle("")
    plt.title("Path weights by method")
    plt.xlabel("Method")
    plt.ylabel("Path weight")
    plt.tight_layout()
    plt.show()


## Notes and references

- `mij_EE_matrix.csv` is used as the input matrix in this notebook. Every row/column label in that file is treated as an excitatory node.
- Diagonal/self entries are ignored in this notebook. The transition matrix has its diagonal set to zero before methods are run, so path scores reflect transitions between distinct excitatory nodes.
- `summed_weight` is the sum of selected positive transition weights between distinct excitatory neurons. Because all methods use the same scoring helper, score differences are attributable to path selection rather than to different objective functions.
- The greedy tree/path follows the largest positive outgoing transition to an unvisited excitatory neuron. It is fast and interpretable, but it can be trapped by locally attractive early choices.
- The maximum spanning tree method is undirected for topology; selected path scores still use the original directed transition weights. This makes it a sparse structural backbone rather than an exact directed path optimizer.
- For 32 excitatory nodes, exact full bitmask DP is usually computationally prohibitive, so the notebook defaults to beam-limited DP unless `NOTEBOOK_EXACT_DP_MAX_E_NODES` is increased.
- Beam-limited DP is not guaranteed to maximize the final path weight; it only maximizes among the states that survive the beam at each expansion step.
- Branch-and-bound is exact only when it finishes with `bb_status == "complete"`; otherwise it reports the best path found under the time/node limits.
- MILP with subtour elimination is exact only when `milp_status == "Optimal"`; otherwise, increase `MILP_TIME_LIMIT_PER_SEED` before using it as the maximizing benchmark.
- The maximum-weight asymmetric Hamiltonian path is exact only when `hamiltonian_status == "Optimal"`; it is forced to visit every excitatory node exactly once, so it solves a stricter complete-ordering problem than the unconstrained MILP.
- Relevant DP computation reference: Held, M., & Karp, R. M. (1962). "A dynamic programming approach to sequencing problems." Journal of the Society for Industrial and Applied Mathematics, 10(1), 196-210. https://doi.org/10.1137/0110015.
- Relevant MILP subtour-elimination reference: Miller, C. E., Tucker, A. W., & Zemlin, R. A. (1960). "Integer Programming Formulation of Traveling Salesman Problems." Journal of the ACM, 7(4), 326-329. https://doi.org/10.1145/321043.321046.
